# dS calculation pipiline

**input**
- `../data/cds.fa` — LM04 CDS sequences
- `../data/outgroup_*.gff` — miniprot results
- `../data/Claviceps_*.fna` — outgroup genomes

**output**
- `../data/ds_alignment/out.tsv` — for each gene, dS / dN / sites

conda envs：numpy / scipy / pandas / mafft / miniprot
---


## 1. envs and parameters

In [ ]:
import collections
import concurrent.futures as cf
import itertools
import os
import re
import subprocess
import sys

import numpy as np
import pandas as pd

# ── dir ──────────────────────────────────────────────────────────
BASE = 'root dir'
CDS_FA   = f'{BASE}/../data/cds.fa'
GFF      = f'{BASE}/../data/*.gff'
GENOME   = f'{BASE}/../data/*.fa'
OUT_DIR  = f'{BASE}/../data/ds_alignment'

MAFFT = '/PATH_to_mafft'

# ── filter values────────────────────────────────────────
MIN_S, MIN_N = 10, 30      # ds / dn site bottom values
DROP_DS      = 1e-3        # dS < 0.001 discarded
MIN_CODON    = 50          # back-translation min codons
N_PROC       = 10

os.makedirs(OUT_DIR, exist_ok=True)
print('OK')

## 2. Fasta reader

In [ ]:
def load_fasta(path):
    """reading FASTA，return {id: seq}"""
    seqs, name, buf = {}, None, []
    for line in open(path):
        if line.startswith('>'):
            if name is not None:
                seqs[name] = ''.join(buf).upper()
            name = line[1:].split()[0]
            buf = []
        else:
            buf.append(line.strip())
    if name is not None:
        seqs[name] = ''.join(buf).upper()
    return seqs


ref = load_fasta(CDS_FA)
print(f'LM04 CDS: {len(ref)} ')
print(f'example: {list(ref)[:3]}')
print(f'length median: {int(np.median([len(s) for s in ref.values()]))} bp')

## 3. codon table


In [ ]:
BASES = 'TCAG'
AAS   = 'FFLLSSSSYY**CC*WLLLLPPPPHHQQRRRRIIIMTTTTNNKKSSRRVVVVAAAADDEEGGGG'
CODON = {''.join(c): a for c, a in zip(itertools.product(BASES, repeat=3), AAS)}
RC    = str.maketrans('ACGTNacgtn', 'TGCANtgcan')

print(f'condon table {len(CODON)} ；TGA = {CODON["TGA"]}，ATG = {CODON["ATG"]}')

## 4. generate outgroup cds from miniprot GFF files

In [ ]:
def build_outgroup_cds(gff_path, genome, min_codon=MIN_CODON):
    """parse miniprot GFF → {LM04 gene ID: outgroup CDS}。
    """
    mrna, cds = {}, collections.defaultdict(list)

    for line in open(gff_path):
        if line.startswith('#'):
            continue
        f = line.rstrip('\n').split('\t')
        if len(f) < 9:
            continue
        attrs = dict(kv.split('=', 1) for kv in f[8].split(';') if '=' in kv)

        if f[2] == 'mRNA':
            mrna[attrs['ID']] = dict(chrom=f[0], strand=f[6],
                                     target=attrs.get('Target', ''))
        elif f[2] == 'CDS':
            # record coordinates
            cds[attrs['Parent']].append((int(f[3]), int(f[4])))


    out = {}
    for mid, segs in cds.items():
        if mid not in mrna:
            continue
        m = mrna[mid]
        # Target = "Cp_000002-T1 1 123" → get Cp_000002
        tgt = m['target'].split()[0].split('-T')[0] if m['target'] else None
        if not tgt or m['chrom'] not in genome:
            continue

        segs.sort()                      
        seq = ''.join(genome[m['chrom']][s - 1:e] for s, e in segs)
        if m['strand'] == '-':
            seq = seq.translate(RC)[::-1]  
        seq = seq[:len(seq) // 3 * 3]     

        if len(seq) >= min_codon * 3:
            out[tgt] = seq
    return out

In [ ]:

genome = load_fasta(GENOME)
og = build_outgroup_cds(GFF, genome)
print(f'outgroup CDS number: {len(og)}    （LM04 CDS number: {len(ref)} ）')
print(f'intersection between LM04 and outgroup: {len(set(ref) & set(og))} ')

In [ ]:
# check 3 genes:coordinates,strand direction,length,any internal stop codons
for g in list(og)[:10]:
    s = og[g]
    prot = ''.join(CODON.get(s[i:i+3], 'X') for i in range(0, len(s) - 2, 3))
    print(f'{g}: CDS {len(s)} bp → {len(prot)} aa，'
          f'initial codons {prot[:1]}，internal stop codons {prot[:-1].count("*")}')

## 5. Translation

In [ ]:
def translate(seq):
    """non ACGT would be labelled as X"""
    return ''.join(CODON.get(seq[i:i+3], 'X') for i in range(0, len(seq) - 2, 3))


# check translation quality
bad = 0
for g, s in og.items():
    p = translate(s)
    if 'X' in p:
        bad += 1
print(f'CDS containing X: {bad} / {len(og)}')

## 6. protein alignment + back-translation

In [ ]:
def mafft_protein(pa, pb):
    """two protein alignment, return (aln_a, aln_b)，which keeps the same length."""
    fa = f'>a\n{pa}\n>b\n{pb}\n'
    r = subprocess.run([MAFFT, '--quiet', '--auto', '-'],
                       input=fa, capture_output=True, text=True)
    if r.returncode != 0:
        return None, None
    seqs, name = {}, None
    #print(r.stdout)
    for line in r.stdout.split('\n'):
        if line.startswith('>'):
            name = line[1:].strip()
            seqs[name] = []
        elif name and line.strip():
            seqs[name].append(line.strip())   
    if 'a' not in seqs or 'b' not in seqs:
        return None, None
    a = ''.join(seqs['a'])
    b = ''.join(seqs['b'])
    return (a, b) if len(a) == len(b) else (None, None)


def back_translate(aln_q, aln_t, cds_q, cds_t):
    """build codon alignment from the above protein alignments, and remove the gaps
    aln_q / aln_t : protein alignment（containing '-'）
    cds_q / cds_t : initial CDS without gaps
    return (codon_q, codon_t) 
    if failure, return (None, None)
    """
    cod_q = [cds_q[i:i+3] for i in range(0, len(cds_q) - 2, 3)]
    cod_t = [cds_t[i:i+3] for i in range(0, len(cds_t) - 2, 3)]

    iq = it = 0
    pairs = []
    for j in range(len(aln_q)):
        if aln_q[j] == '-':
            cq = '---'
        else:
            if iq >= len(cod_q):
                return None, None
            cq = cod_q[iq]
            iq += 1

        if aln_t[j] == '-':
            ct = '---'
        else:
            if it >= len(cod_t):
                return None, None
            ct = cod_t[it]
            it += 1

        if '-' not in cq + ct and 'N' not in cq + ct:
            pairs.append((cq, ct))

    if len(pairs) < MIN_CODON:
        return None, None
    return (''.join(a for a, _ in pairs), ''.join(b for _, b in pairs))


In [ ]:

# case1：pick up a long gene for testing codes
DEMO = next(g for g in ref if g in og and len(og[g]) >= 3000)
print(f'case gene: {DEMO}')
pq, pt = translate(ref[DEMO]), translate(og[DEMO])
aq, at = mafft_protein(pq, pt)
cq, ct = back_translate(aq, at, ref[DEMO], og[DEMO])
print(f'{DEMO}: LM04 {len(pq)} aa，outgroup {len(pt)} aa')
print(f'  protein alignment length {len(aq)}')
print(f'  codon number after back-translation {len(cq)//3}')
print(f'  the first 70 amino acidds of the protein alignment:')
print('   q', aq[:70])
print('   t', at[:70])
print('   cdsq', cq[1499:])
print('   cdst', ct[1499:])

## 7. Nei–Gojobori method + Jukes–Cantor correction


In [ ]:
def syn_sites(codon):
    """return (number of synonymous sites, number of nonsynonymous sites)
       return None for stop codons."""
    if len(codon) != 3 or '-' in codon or 'N' in codon:
        return None
    aa = CODON.get(codon, '*')
    if aa == '*':
        return None

    s = 0.0
    for i in range(3):
        for b in 'ACGT':
            if b == codon[i]:
                continue
            alt = codon[:i] + b + codon[i+1:]
            if CODON.get(alt) == '*':    
                continue
            if CODON[alt] == aa:    
                s += 1 / 3
    return s, 3 - s


def codon_diffs(c1, c2):
    """return (synonymous differences, nonsynonymous differences)
       multi-base differences are averaged over all paths."""
    diffs = [i for i in range(3) if c1[i] != c2[i]]
    if not diffs:
        return 0.0, 0.0

    if len(diffs) == 1:
        i = diffs[0]
        alt = c1[:i] + c2[i] + c1[i+1:]
        return (1.0, 0.0) if CODON[alt] == CODON[c1] else (0.0, 1.0)

    s_tot = n_tot = 0.0
    n_path = 0
    for perm in itertools.permutations(diffs):
        cur, s, n, ok = c1, 0, 0, True
        for i in perm:
            alt = cur[:i] + c2[i] + cur[i+1:]
            if CODON.get(alt) == '*':      
                ok = False
                break
            if CODON[alt] == CODON[cur]:
                s += 1
            else:
                n += 1
            cur = alt
        if ok:
            s_tot += s
            n_tot += n
            n_path += 1
    if n_path == 0:
        return 0.0, 0.0
    return s_tot / n_path, n_tot / n_path


def jukes_cantor(p):
    """JC69 correction. p ≥ 0.75 → saturation, return None"""
    if p is None or p >= 0.75:
        return None
    return -0.75 * np.log(1 - 4 * p / 3) if p > 0 else 0.0


# case2:
for c in ('TTT', 'TTC', 'TTA', 'CTA', 'ATG', 'TGG'):
    print(f'{c} ({CODON[c]}): synonymous sites {syn_sites(c)[0]:.2f} / non-synonymous sites {syn_sites(c)[1]:.2f}')
print()
print('TTT→TTC (single-base synonymous):', codon_diffs('TTT', 'TTC'))
print('TTT→GCA (triple bases    ):', codon_diffs('TTT', 'GCA'))
print('p=0.1 → dS =', round(jukes_cantor(0.1), 5))
print('p=0.5 → dS =', round(jukes_cantor(0.5), 5))

print(codon_diffs("ACT", "AGG"))


## 8. calculating dS for individual gene


In [ ]:
def pairwise_ds(cds_q, cds_t, min_s=MIN_S, min_n=MIN_N):
    """dS / dN, return dict or None"""
    pq, pt = translate(cds_q), translate(cds_t)
    if 'X' in pq or 'X' in pt:
        return None

    aq, at = mafft_protein(pq, pt)
    if aq is None:
        return None

    cq, ct = back_translate(aq, at, cds_q, cds_t)
    if cq is None:
        return None

    S = N = Sd = Nd = 0.0
    for i in range(0, len(cq), 3):
        c1, c2 = cq[i:i+3], ct[i:i+3]
        r = syn_sites(c1)
        if r is None or CODON.get(c2, '*') == '*':
            continue
        S += r[0]
        N += r[1]
        s, n = codon_diffs(c1, c2)
        Sd += s
        Nd += n

    if S < min_s or N < min_n:            # few sites
        return None
    ds = jukes_cantor(Sd / S)
    dn = jukes_cantor(Nd / N)
    if ds is None or dn is None or ds <= DROP_DS:
        return None
    return dict(dS=ds, dN=dn, S=S, N=N, Sd=Sd, Nd=Nd)


r = pairwise_ds(ref[DEMO], og[DEMO])
print(f'{DEMO}:')
print(f'  synonymous S = {r["S"]:.0f}，synonymous differences Sd = {r["Sd"]:.0f} → pS = {r["Sd"]/r["S"]:.4f}')
print(f'  non-synonymous N = {r["N"]:.0f}，non-synonymous differences Nd = {r["Nd"]:.0f} → pN = {r["Nd"]/r["N"]:.4f}')
print(f'  dS = {r["dS"]:.4f}   dN = {r["dN"]:.4f}   ω = {r["dN"]/r["dS"]:.3f}')

## 9. batch calculation for the whole genome


In [ ]:
def _worker(args):
    gene, cq, ct = args
    try:
        r = pairwise_ds(cq, ct)
        return gene, r, 'ok' if r else 'filtered'
    except Exception as exc:
        return gene, None, f'err:{type(exc).__name__}'


jobs = [(g, ref[g], og[g]) for g in ref if g in og]
print(f'job number: {len(jobs)}  ...')

rows, why = [], collections.Counter()
with cf.ProcessPoolExecutor(max_workers=N_PROC) as ex:
    for g, r, w in ex.map(_worker, jobs, chunksize=20):
        why[w] += 1
        if r:
            rows.append(dict(gene=g, **r))

df = pd.DataFrame(rows)
print(f'Finished number：{len(df)} ')
print(f'state explanation：{dict(why)}')

In [ ]:
df[['dS', 'dN', 'S', 'N', 'Sd', 'Nd']].describe().round(4)

## 12. Summary statistics and window aggregation


In [ ]:
print(f'calculated gene number: {len(df)}')
print(f'dS median {df.dS.median():.4f}   P90 {df.dS.quantile(0.9):.4f}')
print(f'dS ≥ 1 percentage: {100*(df.dS >= 1).mean():.1f}%  ')
print(f'ω median {(df.dN / df.dS).median():.3f}')

In [ ]:
WIN = 50_000

gene_pos = {}
for line in open(f'{BASE}/../data/LM04.funannotate.gff3'):
    if line.startswith('#'):
        continue
    f = line.rstrip('\n').split('\t')
    if len(f) < 9 or f[2] != 'gene':
        continue
    g = f[8].split('ID=')[1].split(';')[0] if 'ID=' in f[8] else None
    if g:
        gene_pos[g] = (f[0], int(f[3]))

win = collections.defaultdict(list)
for _, row in df.iterrows():
    g = row['gene']
    if g not in gene_pos:
        continue
    chrom, start = gene_pos[g]
    win[f'{chrom}:{(start-1)//WIN}'].append(row['dS'])

win_ds = {k: float(np.median(v)) for k, v in win.items() if len(v) >= 3}
print(f'window number {len(win_ds)}（≥ 3 genes）')
print(f'window dS median {np.median(list(win_ds.values())):.4f}')

## 13. writing CSV

output format：`gene  dS  dN  S_sites  N_sites  S_diff  N_diff`


In [ ]:
out_path = f'{OUT_DIR}/outgroup_**.tsv'
df_out = df[['gene', 'dS', 'dN', 'S', 'N', 'Sd', 'Nd']].rename(
    columns={'S': 'S_sites', 'N': 'N_sites', 'Sd': 'S_diff', 'Nd': 'N_diff'})
df_out.to_csv(out_path, sep='\t', index=False, float_format='%.6f')
print(f'-> {out_path}  ({len(df_out)} )')
print(df_out.head())